# 02: NLP Review Sentiment & Aspect-Based Sentiment Analysis (ABSA)
### E-commerce Customer Intelligence & Personalization Platform
This notebook trains TF-IDF + Logistic Regression and Naive Bayes sentiment models on held-out test splits, extracts fine-grained aspect sentiments across 11 product dimensions, and clusters customer complaint themes via NMF.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.nlp.sentiment_train import train_sentiment_models
from src.nlp.aspect_extraction import extract_aspect_sentiment_from_text, process_reviews_absa
from src.nlp.topic_modeling import train_topic_model
from src.utils.config import PROCESSED_DATA_DIR, MODELS_SENTIMENT_DIR
sns.set_theme(style='whitegrid')

## 1. Train & Evaluate Sentiment Classifier (No Data Leakage)

In [ ]:
clf, vec, eval_results = train_sentiment_models()
print('Sentiment Classifier Accuracy:', eval_results['accuracy'])
print('Naive Bayes Baseline Accuracy:', eval_results['baseline_accuracy'])

report_df = pd.DataFrame(eval_results['classification_report']).transpose()
print('\nClassification Report:')
display(report_df)

## 2. Confusion Matrix Heatmap

In [ ]:
cm = np.array(eval_results['confusion_matrix'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=eval_results['classes'], yticklabels=eval_results['classes'])
plt.title('Sentiment Prediction Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Ground Truth Label')
plt.show()

## 3. Aspect-Based Sentiment Analysis (ABSA) across 11 Aspects

In [ ]:
aspects_df = pd.read_csv(PROCESSED_DATA_DIR / 'review_aspects.csv')
asp_summary = aspects_df.groupby('aspect').agg(
    mentions=('review_id', 'count'),
    avg_score=('sentiment_score', 'mean'),
    pos_pct=('sentiment', lambda s: (s == 'Positive').mean() * 100)
).reset_index().sort_values('avg_score', ascending=False)

plt.figure(figsize=(10, 5))
colors = ['#10b981' if s > 0 else '#ef4444' for s in asp_summary['avg_score']]
plt.barh(asp_summary['aspect'], asp_summary['avg_score'], color=colors)
plt.title('Average Aspect Sentiment Polarity Score (-1.0 to +1.0)')
plt.xlabel('Mean Sentiment Score')
plt.show()